In [3]:
%pip install yfinance


[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



  Using cached multitasking-0.0.12-py3-none-any.whl
  Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl.metadata (13 kB)
  Using cached cffi-2.0.0-cp312-cp312-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-2.23-py3-none-any.whl.metadata (993 bytes)
Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl (1.6 MB)
Using cached cffi-2.0.0-cp312-cp312-win_amd64.whl (183 kB)
Using cached pycparser-2.23-py3-none-any.whl (118 kB)


In [ ]:
import requests
import pandas as pd
import numpy as np
import time
from datetime import datetime
import pytz

# ================= CONFIG =================
BOT_TOKEN = "7699036883:AAEh1PxEVSoqaYyto0E1yByjgxC4q5mLeJw"
CHAT_ID = "1155443179"

# =============== BASE AGENT =================
class BaseAgent:
    def run(self, *args, **kwargs):
        raise NotImplementedError

# =============== DATA FETCH AGENT =================
class DataAgent(BaseAgent):
    def __init__(self, symbol="XAUUSDT", interval="1m", limit=300):
        self.symbol = symbol
        self.interval = interval
        self.limit = limit

    def run(self):
        # CHANGED: Using Futures API because XAUUSDT is a futures contract on Binance
        url = "https://fapi.binance.com/fapi/v1/klines"
        params = {
            "symbol": self.symbol,
            "interval": self.interval,
            "limit": self.limit
        }
        
        try:
            response = requests.get(url, params=params)
            response.raise_for_status() # Check for errors
            data = response.json()
            
            df = pd.DataFrame(data, columns=[
                "timestamp","open","high","low","close","volume",
                "close_time","quote_asset_volume","num_trades",
                "taker_buy_base","taker_buy_quote","ignore"
            ])

            df["close"] = df["close"].astype(float)
            df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
            return df[["timestamp", "close"]]
            
        except Exception as e:
            print(f"Error fetching data: {e}")
            return pd.DataFrame() # Return empty DF on error

# =============== RSI AGENT =================
class RSIAgent(BaseAgent):
    def __init__(self, window=14):
        self.window = window

    def run(self, df: pd.DataFrame):
        if df.empty: return df
        
        delta = df["close"].diff()
        gain = delta.where(delta > 0, 0.0)
        loss = -delta.where(delta < 0, 0.0)

        avg_gain = gain.ewm(alpha=1/self.window, min_periods=self.window).mean()
        avg_loss = loss.ewm(alpha=1/self.window, min_periods=self.window).mean()

        rs = avg_gain / avg_loss
        df["rsi"] = 100 - (100 / (1 + rs))
        return df

# =============== EMA 200 AGENT =================
class EMAAgent(BaseAgent):
    def __init__(self, span=200):
        self.span = span

    def run(self, df: pd.DataFrame):
        if df.empty: return df
        df["ema_200"] = df["close"].ewm(span=self.span, adjust=False).mean()
        return df

# =============== ALERT AGENT =================
class AlertAgent(BaseAgent):
    def __init__(self, bot_token, chat_id):
        self.bot_token = bot_token
        self.chat_id = chat_id
        self.ist = pytz.timezone("Asia/Kolkata")

    def send_message(self, message: str):
        url = f"https://api.telegram.org/bot{self.bot_token}/sendMessage"
        payload = {"chat_id": self.chat_id, "text": message}
        try:
            requests.post(url, json=payload)
        except Exception as e:
            print(f"Error sending msg: {e}")

    def run(self, df: pd.DataFrame, symbol="XAUUSDT"):
        if df.empty: return

        latest = df.iloc[-1]

        price = latest["close"]
        rsi = latest["rsi"]
        ema_200 = latest["ema_200"]

        now_ist = datetime.now(self.ist).strftime("%d-%m-%Y / %I:%M %p")

        if pd.notna(rsi) and pd.notna(ema_200):
            
            # Gold prices usually have 2 decimals, formatted accordingly
            msg_body = (
                f"Price: {price:.2f}\n"
                f"EMA 200: {ema_200:.2f}\n"
                f"RSI: {rsi:.2f}\n"
                f"🕒 {now_ist}"
            )

            # BUY SIGNAL
            if price > ema_200 and rsi <= 30:
                self.send_message(f"🟢 GOLD BUY SIGNAL - {symbol}\n{msg_body}")
                print(f"🟢 BUY SIGNAL SENT FOR {symbol}")

            # SELL SIGNAL
            elif price < ema_200 and rsi >= 70:
                self.send_message(f"🔴 GOLD SELL SIGNAL - {symbol}\n{msg_body}")
                print(f"🔴 SELL SIGNAL SENT FOR {symbol}")
                
            else:
                print(
                    f"No signal {symbol} | "
                    f"Price={price:.2f}, EMA={ema_200:.2f}, RSI={rsi:.2f}"
                )

# =============== ORCHESTRATOR =================
class OrchestratorAgent(BaseAgent):
    def __init__(self, symbols):
        self.symbols = symbols
        # Defaulting DataAgent to 1m interval
        self.data_agent = DataAgent(interval="1m")
        self.rsi_agent = RSIAgent()
        self.ema_agent = EMAAgent()
        self.alert_agent = AlertAgent(BOT_TOKEN, CHAT_ID)

    def run(self):
        for symbol in self.symbols:
            self.data_agent.symbol = symbol
            df = self.data_agent.run()
            
            if not df.empty:
                df = self.rsi_agent.run(df)
                df = self.ema_agent.run(df)
                self.alert_agent.run(df, symbol)

# =============== MAIN LOOP =================
if __name__ == "__main__":
    # Ensure this symbol exists on Binance Futures
    symbols = ["XAUUSDT"]

    print("🤖 Gold Bot (1-Min) Started...")
    orchestrator = OrchestratorAgent(symbols)

    while True:
        try:
            orchestrator.run()
            # Wait 60 seconds for the next 1-minute candle
            time.sleep(60)
        except KeyboardInterrupt:
            print("Bot stopped by user.")
            break
        except Exception as e:
            print(f"Main loop error: {e}")
            time.sleep(60)

🤖 Gold Bot (1-Min) Started...
No signal XAUUSDT | Price=4509.63, EMA=4510.14, RSI=47.99
No signal XAUUSDT | Price=4509.82, EMA=4510.13, RSI=50.22
No signal XAUUSDT | Price=4509.82, EMA=4510.13, RSI=50.22
No signal XAUUSDT | Price=4510.24, EMA=4510.12, RSI=54.62


In [1]:
import pandas as pd

In [3]:
df = pd.ExcelFile(r'E:\TRADING\test.xlsx', engine='openpyxl')

In [7]:
df1 = pd.read_excel(df, sheet_name='Sheet1', usecols='A:C', nrows=8)
df1

,id,name,department
0,1,A,a
1,2,B,a
2,2,C,d
3,3,D,d
4,4,E,d
5,5,S,b
6,6,A,a
7,7,R,a


In [9]:
df2 = pd.read_excel(df, sheet_name='Sheet1', usecols='E:G', nrows=11)
df2

,id.1,name.1,department.1
0,1,A,a
1,2,B,a
2,2,C,d
3,3,D,d
4,4,E,d
5,5,S,b
6,6,A,a
7,7,R,a
8,8,G,d
9,9,A,d
